In [1]:
import pandas as pd

In [2]:
likes = pd.read_csv('likes.csv')
views = pd.read_csv('views.csv')

In [3]:
likes.head()

,user_id,post_id,timestamp
0,128381,4704,1654030804
1,146885,1399,1654030816
2,50948,2315,1654030828
3,14661,673,1654030831
4,37703,1588,1654030833


In [4]:
views.head()

,user_id,exp_group,recommendations,timestamp
0,128381,control,[3644 4529 4704 5294 4808],1654030803
1,146885,test,[1399 1076 797 7015 5942],1654030811
2,50948,test,[2315 3037 1861 6567 4093],1654030825
3,37703,test,[2842 1949 162 1588 6794],1654030826
4,14661,test,[2395 5881 5648 3417 673],1654030829


In [13]:
result = views.groupby('user_id').filter(lambda x: x['exp_group'].nunique() > 1)

In [33]:
user_ids = result['user_id'].unique().tolist()

In [36]:
print(len(likes))
print(len(views))
print(len(likes[likes['user_id'].isin(user_ids)]))
print(len(views[views['user_id'].isin(user_ids)]))

230176
193295
39
27


In [37]:
likes_m = likes[~likes['user_id'].isin(user_ids)]
views_m = views[~views['user_id'].isin(user_ids)]

print(len(likes_m))
print(len(views_m))

230137
193268


In [41]:
user_gr = views[['user_id','exp_group']].drop_duplicates().reset_index(drop=True)

In [45]:
user_gr['exp_group'].value_counts(normalize=True)

exp_group
test       0.502376
control    0.497624
Name: proportion, dtype: float64

In [46]:
from scipy.stats import binomtest

counts = user_gr['exp_group'].value_counts()
n_test = counts['test']
n_total = counts.sum()

# Проверяем соответствие доли ожидаемым 0.5 (двусторонний тест)
res = binomtest(k=n_test, n=n_total, p=0.5, alternative='two-sided')
print(f"p-value: {res.pvalue:.4f}")

p-value: 0.2271


In [48]:
n_total

np.int64(65017)

In [ ]:
unique_user_views = views_m['user_id'].drop_duplicates().tolist()


In [124]:
len(unique_user_views)

65009

In [58]:
min_time = views_m['timestamp'].min()
max_time = views_m['timestamp'].max()

In [65]:
user_likes_ = likes_m[(likes_m['timestamp']>=min_time) & (likes_m['timestamp']<=max_time) & (likes_m['user_id'].isin(unique_user_views))]

In [66]:
user_likes_count = len(user_likes_['user_id'].drop_duplicates().tolist())

In [68]:
user_likes_count / len(unique_user_views)

0.8946607392822532

In [72]:

user_data = pd.merge(user_likes_, views_m[['user_id','exp_group']], how = 'left', on = 'user_id')

In [76]:
user_data

,user_id,post_id,timestamp,exp_group
0,128381,4704,1654030804,control
1,128381,4704,1654030804,control
2,128381,4704,1654030804,control
3,128381,4704,1654030804,control
4,146885,1399,1654030816,test
...,...,...,...,...
1006050,52169,1709,1655240369,test
1006051,52169,1709,1655240369,test
1006052,41078,1540,1655240370,control
1006053,41078,1540,1655240370,control


In [79]:
likes_agg = (
    user_likes_
    .groupby('user_id')
    .size()
    .reset_index(name='like_count')
)

In [80]:
users_unique  = user_data[['user_id', 'exp_group']].drop_duplicates()
user_data_up = pd.merge(users_unique, likes_agg, on='user_id', how='left')

In [81]:
user_data_up

,user_id,exp_group,like_count
0,128381,control,7
1,146885,test,4
2,50948,test,5
3,14661,test,11
4,37703,test,6
...,...,...,...
58156,80500,test,1
58157,24891,control,1
58158,119630,test,1
58159,52169,test,1


In [82]:
from scipy.stats import mannwhitneyu

group_test = user_data_up[user_data_up['exp_group']=='test']['like_count']
group_control = user_data_up[user_data_up['exp_group']=='control']['like_count']
_, p_value = mannwhitneyu(group_test, group_control, alternative='two-sided')

In [83]:
p_value

np.float64(0.0017969598558867396)

In [84]:
from statsmodels.stats.proportion import proportions_ztest

# 1. Проверяем направление роста для первой метрики (число лайков на юзера)
print(f"Среднее лайков -> Control: {group_control.mean():.4f} | Test: {group_test.mean():.4f}")

# 2. Считаем вторую метрику: доля пользователей с хотя бы 1 лайком
user_data_up['has_like'] = (user_data_up['like_count'] > 0).astype(int)

test_has_like = user_data_up[user_data_up['exp_group'] == 'test']['has_like']
ctrl_has_like = user_data_up[user_data_up['exp_group'] == 'control']['has_like']

_, p_val_z = proportions_ztest(
    count=[test_has_like.sum(), ctrl_has_like.sum()],
    nobs=[len(test_has_like), len(ctrl_has_like)]
)

print(f"Доля с лайком  -> Control: {ctrl_has_like.mean():.4f} | Test: {test_has_like.mean():.4f} | p-value: {p_val_z:.6f}")


Среднее лайков -> Control: 3.9115 | Test: 3.9989
Доля с лайком  -> Control: 1.0000 | Test: 1.0000 | p-value: nan


e:\python_projects\ML_course_karpov\env\Lib\site-packages\statsmodels\stats\weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std


In [85]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest

# 1. Полный список всех пользователей эксперимента (включая тех, кто не поставил ни одного лайка)
all_users = views_m[['user_id', 'exp_group']].drop_duplicates()

# 2. Считаем количество уникальных лайков на каждого пользователя из таблицы likes
likes_per_user = (
    likes.drop_duplicates(subset=['user_id', 'post_id', 'timestamp'])
    .groupby('user_id')
    .size()
    .reset_index(name='like_count')
)

# 3. Объединяем и заполняем 0 для пользователей без лайков
user_data_fixed = pd.merge(all_users, likes_per_user, on='user_id', how='left')
user_data_fixed['like_count'] = user_data_fixed['like_count'].fillna(0).astype(int)
user_data_fixed['has_like'] = (user_data_fixed['like_count'] > 0).astype(int)

# 4. Делим на группы и считаем оба теста заново
test_df = user_data_fixed[user_data_fixed['exp_group'] == 'test']
ctrl_df = user_data_fixed[user_data_fixed['exp_group'] == 'control']

# Тест 1: Число лайков на пользователя (Манна-Уитни)
_, p_val_mw = mannwhitneyu(test_df['like_count'], ctrl_df['like_count'], alternative='two-sided')

# Тест 2: Доля пользователей с хотя бы 1 лайком (z-критерий)
_, p_val_z = proportions_ztest(
    count=[test_df['has_like'].sum(), ctrl_df['has_like'].sum()],
    nobs=[len(test_df), len(ctrl_df)]
)

print(f"Всего юзеров   -> Control: {len(ctrl_df)} | Test: {len(test_df)}")
print(f"Среднее лайков -> Control: {ctrl_df['like_count'].mean():.4f} | Test: {test_df['like_count'].mean():.4f} | p-value: {p_val_mw:.6f}")
print(f"Доля с лайком  -> Control: {ctrl_df['has_like'].mean():.4f} | Test: {test_df['has_like'].mean():.4f} | p-value: {p_val_z:.6f}")


Всего юзеров   -> Control: 32350 | Test: 32659
Среднее лайков -> Control: 3.4871 | Test: 3.5926 | p-value: 0.000030
Доля с лайком  -> Control: 0.8913 | Test: 0.8982 | p-value: 0.004455


А теперь давайте сделаем самое сложное — посчитаем hitrate (или долю рекомендаций, в которые пользователи кликнули). 

Сделать join по пользователям всех лайков и всех показов рекомендаций (то есть у нас появятся пары всех показов и всех лайков одного и того же пользователя, но не все эти пары реальны). Далее отфильтровать те ситуации, где лайк предшествовал показу рекомендаций (то есть относился на самом деле к другому показу) или был слишком поздно (тоже относится к другому показу).

In [113]:
user_data_all = pd.merge(likes_m, views_m,  on='user_id', how='left')

In [114]:
user_data_all

,user_id,post_id,timestamp_x,exp_group,recommendations,timestamp_y
0,128381,4704,1654030804,control,[3644 4529 4704 5294 4808],1654030803
1,128381,4704,1654030804,control,[ 28 1433 2870 5544 1585],1654458854
2,128381,4704,1654030804,control,[3029 3608 2542 4165 3490],1655049326
3,128381,4704,1654030804,control,[2573 193 5149 7007 2430],1655066663
4,146885,1399,1654030816,test,[1399 1076 797 7015 5942],1654030811
...,...,...,...,...,...,...
1006353,13267,1787,1655243692,test,[3687 3542 1727 196 4747],1654584391
1006354,13267,1787,1655243692,test,[1939 1546 5482 1220 3259],1655225687
1006355,13267,1787,1655243692,test,[1787 6033 7047 1491 247],1655240212
1006356,67302,7255,1655243736,test,[6960 1906 1662 5416 6694],1654701056


In [115]:
user_data_all=user_data_all.rename(columns={'timestamp_x': 'timestamp_likes'})
user_data_all=user_data_all.rename(columns={'timestamp_y': 'timestamp_views'})

In [116]:
user_data_all

,user_id,post_id,timestamp_likes,exp_group,recommendations,timestamp_views
0,128381,4704,1654030804,control,[3644 4529 4704 5294 4808],1654030803
1,128381,4704,1654030804,control,[ 28 1433 2870 5544 1585],1654458854
2,128381,4704,1654030804,control,[3029 3608 2542 4165 3490],1655049326
3,128381,4704,1654030804,control,[2573 193 5149 7007 2430],1655066663
4,146885,1399,1654030816,test,[1399 1076 797 7015 5942],1654030811
...,...,...,...,...,...,...
1006353,13267,1787,1655243692,test,[3687 3542 1727 196 4747],1654584391
1006354,13267,1787,1655243692,test,[1939 1546 5482 1220 3259],1655225687
1006355,13267,1787,1655243692,test,[1787 6033 7047 1491 247],1655240212
1006356,67302,7255,1655243736,test,[6960 1906 1662 5416 6694],1654701056


In [117]:
time_diff = user_data_all['timestamp_likes'] - user_data_all['timestamp_views']
is_time_ok = (time_diff >= 0) & (time_diff <= 3600)
def post_in_recs(row):
    if pd.isna(row['post_id']) or pd.isna(row['recommendations']):
        return False
    
    recs = row['recommendations']
    target_post = str(int(row['post_id']))
    
    # Если колонка — строка вида "[3644 4529 4704 5294 4808]" или "[3644, 4529, ...]"
    if isinstance(recs, str):
        tokens = recs.replace('[', '').replace(']', '').replace(',', ' ').split()
        return target_post in tokens
    
    # Если вдруг там список или numpy-массив
    return int(row['post_id']) in recs

is_post_ok = user_data_all.apply(post_in_recs, axis=1)

# Итоговый флаг: является ли пара показ-лайк реальной
user_data_all['is_valid_like'] = is_time_ok & is_post_ok

In [118]:
user_data_all

,user_id,post_id,timestamp_likes,exp_group,recommendations,timestamp_views,is_valid_like
0,128381,4704,1654030804,control,[3644 4529 4704 5294 4808],1654030803,True
1,128381,4704,1654030804,control,[ 28 1433 2870 5544 1585],1654458854,False
2,128381,4704,1654030804,control,[3029 3608 2542 4165 3490],1655049326,False
3,128381,4704,1654030804,control,[2573 193 5149 7007 2430],1655066663,False
4,146885,1399,1654030816,test,[1399 1076 797 7015 5942],1654030811,True
...,...,...,...,...,...,...,...
1006353,13267,1787,1655243692,test,[3687 3542 1727 196 4747],1654584391,False
1006354,13267,1787,1655243692,test,[1939 1546 5482 1220 3259],1655225687,False
1006355,13267,1787,1655243692,test,[1787 6033 7047 1491 247],1655240212,True
1006356,67302,7255,1655243736,test,[6960 1906 1662 5416 6694],1654701056,False


In [125]:
len(user_data_all[user_data_all['is_valid_like']==True])/len(unique_user_views)

3.5401098309464842

In [126]:
views_clean = views_m.copy().reset_index(drop=True)
views_clean['view_id'] = views_clean.index

# 2. Делаем left join всех показов со всеми лайками по user_id
merged = pd.merge(
    views_clean,
    likes_m,  # ваша таблица лайков за период эксперимента
    on='user_id',
    how='left',
    suffixes=('_views', '_likes')
)

print(f"Исходное число показов: {len(views_clean)}")
print(f"Число строк после join: {len(merged)}")

Исходное число показов: 193268
Число строк после join: 1016889


In [127]:
# Проверяем разницу во времени (в секундах)
time_diff = merged['timestamp_likes'] - merged['timestamp_views']
is_time_ok = (time_diff >= 0) & (time_diff <= 3600)

# Проверяем, что post_id есть в списке recommendations (работает для list, np.ndarray и строк)
def post_in_recs(row):
    if pd.isna(row['post_id']) or pd.isna(row['recommendations']):
        return False
    
    recs = row['recommendations']
    target_post = str(int(row['post_id']))
    
    # Если колонка — строка вида "[3644 4529 4704 5294 4808]" или "[3644, 4529, ...]"
    if isinstance(recs, str):
        tokens = recs.replace('[', '').replace(']', '').replace(',', ' ').split()
        return target_post in tokens
    
    # Если вдруг там список или numpy-массив
    return int(row['post_id']) in recs

is_post_ok = merged.apply(post_in_recs, axis=1)

# Итоговый флаг: является ли пара показ-лайк реальной
merged['is_valid_like'] = is_time_ok & is_post_ok

In [128]:
# Считаем количество валидных лайков для каждого показа
views_with_likes = (
    merged.groupby(['view_id', 'user_id', 'exp_group', 'timestamp_views'], as_index=False)
    .agg(likes_count=('is_valid_like', 'sum'))
)

# Флаг: был ли в показе хотя бы один лайк
views_with_likes['has_like'] = (views_with_likes['likes_count'] > 0).astype(int)

print(f"Итоговое число показов после агрегации: {len(views_with_likes)}")
print(f"Общая доля показов с хотя бы одним лайком: {views_with_likes['has_like'].mean():.4f}")

# Доля показов с лайком в разрезе групп эксперимента
print(views_with_likes.groupby('exp_group')['has_like'].agg(['count', 'mean']))


Итоговое число показов после агрегации: 193268
Общая доля показов с хотя бы одним лайком: 0.7133
           count      mean
exp_group                 
control    96370  0.706651
test       96898  0.719829


In [129]:
import hashlib
import pandas as pd
from scipy import stats

def get_bucket(user_id: int, num_buckets: int = 100) -> int:
    # Хэширование гарантирует равномерное распределение без привязки к структуре ID
    hash_val = int(hashlib.md5(f"{user_id}_bucket_salt".encode()).hexdigest(), 16)
    return hash_val % num_buckets

views_with_likes['bucket'] = views_with_likes['user_id'].apply(get_bucket)

In [130]:
bucket_df = (
    views_with_likes
    .groupby(['exp_group', 'bucket'], as_index=False)
    .agg(
        total_views=('has_like', 'count'),
        liked_views=('has_like', 'sum'),
        hitrate=('has_like', 'mean')
    )
)

control_buckets = bucket_df[bucket_df['exp_group'] == 'control']['hitrate']
test_buckets = bucket_df[bucket_df['exp_group'] == 'test']['hitrate']

print(f"Количество бакетов -> Control: {len(control_buckets)}, Test: {len(test_buckets)}")
print(f"Средний Hitrate по бакетам -> Control: {control_buckets.mean():.4f} | Test: {test_buckets.mean():.4f}")


Количество бакетов -> Control: 100, Test: 100
Средний Hitrate по бакетам -> Control: 0.7068 | Test: 0.7197


In [131]:
# 1. t-критерий Стьюдента (основной для бакетного анализа)
t_stat, p_val_t = stats.ttest_ind(test_buckets, control_buckets, equal_var=False)

# 2. Критерий Манна-Уитни (дополнительная проверка на бакетах)
u_stat, p_val_mw = stats.mannwhitneyu(test_buckets, control_buckets, alternative='two-sided')

print(f"t-test (Уэлча) -> stat: {t_stat:.4f}, p-value: {p_val_t:.6f}")
print(f"Mann-Whitney   -> stat: {u_stat:.4f}, p-value: {p_val_mw:.6f}")


t-test (Уэлча) -> stat: 5.4906, p-value: 0.000000
Mann-Whitney   -> stat: 7205.5000, p-value: 0.000000


In [132]:
from scipy import stats

# 1. Hitrate ДО бакетирования (общая доля показов с лайком в каждой группе)
hr_before = views_with_likes.groupby('exp_group')['has_like'].mean()
diff_pp = (hr_before['test'] - hr_before['control']) * 100

print(f"Hitrate control (до бакетов): {hr_before['control']:.4f}")
print(f"Hitrate test    (до бакетов): {hr_before['test']:.4f}")
print(f"Разница (test - control)    : {diff_pp:+.2f} п.п.")

# 2. Проверка значимости ПОСЛЕ бакетирования (100 бакетов)
bucket_hr = views_with_likes.groupby(['exp_group', 'bucket'])['has_like'].mean().unstack(level=0)
_, p_val = stats.ttest_ind(bucket_hr['test'], bucket_hr['control'], equal_var=False)
print(f"p-value (t-test по бакетам) : {p_val:.6f}")


Hitrate control (до бакетов): 0.7067
Hitrate test    (до бакетов): 0.7198
Разница (test - control)    : +1.32 п.п.
p-value (t-test по бакетам) : 0.000000


In [133]:
views_clean = views_m.copy().reset_index(drop=True)
views_clean['view_id'] = views_clean.index

# 2. Делаем left join всех показов со всеми лайками по user_id
merged = pd.merge(
    views_clean,
    likes_m,  # ваша таблица лайков за период эксперимента
    on='user_id',
    how='left',
    suffixes=('_views', '_likes')
)

print(f"Исходное число показов: {len(views_clean)}")
print(f"Число строк после join: {len(merged)}")

merged['bucket'] = merged['user_id'].apply(
    lambda x: int(hashlib.md5((str(x) + 'my_salt').encode()).hexdigest(), 16) % 100
)

Исходное число показов: 193268
Число строк после join: 1016889


In [134]:
merged

,user_id,exp_group,recommendations,timestamp_views,view_id,post_id,timestamp_likes,bucket
0,128381,control,[3644 4529 4704 5294 4808],1654030803,0,4704.0,1.654031e+09,32
1,128381,control,[3644 4529 4704 5294 4808],1654030803,0,5294.0,1.654031e+09,32
2,128381,control,[3644 4529 4704 5294 4808],1654030803,0,3608.0,1.655049e+09,32
3,128381,control,[3644 4529 4704 5294 4808],1654030803,0,2542.0,1.655049e+09,32
4,128381,control,[3644 4529 4704 5294 4808],1654030803,0,4165.0,1.655053e+09,32
...,...,...,...,...,...,...,...,...
1016884,52169,test,[1368 1709 1616 798 5305],1655240354,193265,1709.0,1.655240e+09,73
1016885,142402,test,[5895 6984 1978 6548 6106],1655240373,193266,6548.0,1.655240e+09,68
1016886,72259,control,[6117 1255 6567 3587 3811],1655240388,193267,1712.0,1.654305e+09,28
1016887,72259,control,[6117 1255 6567 3587 3811],1655240388,193267,5070.0,1.654309e+09,28


А теперь давайте оценим различие между группами и значимость. z-критерий для долей мы здесь не можем применять, так как у нас в каждой выборке один и тот же пользователь может встречаться несколько раз. Давайте применим бакетный подход (то есть перейдём к бакетам и по ним оценим значимость), чтобы посчитать групповой hitrate (или CTR) — доля hitrate по группе/бакету. Используйте 100 бакетов. Уровень значимости останется тем же на уровне 0.05.

In [140]:


new_df = views_with_likes.groupby(['exp_group', 'bucket']).agg(
    total_clicks=('has_like', 'count'),
    total_views=('has_like', 'sum'),
    mean_user_ctr=('has_like', 'mean')  # Способ 2: средний CTR пользователей в бакете
).reset_index()

# Способ 1: групповой CTR (отношение суммы кликов к сумме показов в бакете)
new_df['group_ctr'] = new_df['total_clicks'] / new_df['total_views']

# 5. Считаем среднеквадратичное отклонение (std) для одной из групп (например, group == True)
group_true = new_df[new_df['exp_group'] == 'control']

std_group_ctr = group_true['group_ctr'].std()
std_mean_user_ctr = group_true['mean_user_ctr'].std()

group_false = new_df[new_df['exp_group'] == 'test']
std_group_ctr_test = group_false['group_ctr'].std()
std_mean_user_ctr_test = group_false['mean_user_ctr'].std()

print(f"STD группового CTR (sum_clicks / sum_views) control: {std_group_ctr:.5f}")
print(f"STD среднего пользовательского CTR:          {std_mean_user_ctr:.5f}")

print(f"STD группового CTR (sum_clicks / sum_views) test: {std_group_ctr_test:.5f}")
print(f"STD среднего пользовательского CTR:          {std_mean_user_ctr_test:.5f}")

STD группового CTR (sum_clicks / sum_views) control: 0.03110
STD среднего пользовательского CTR:          0.01550
STD группового CTR (sum_clicks / sum_views) test: 0.03476
STD среднего пользовательского CTR:          0.01775


In [139]:
_, p_val = stats.ttest_ind(new_df[new_df['exp_group'] == 'test']['mean_user_ctr'], new_df[new_df['exp_group'] == 'control']['mean_user_ctr'], equal_var=False)
print(f"p-value (t-test по бакетам) : {p_val:.6f}")


p-value (t-test по бакетам) : 0.000000
